# 01 - PyTorch 最小必备

本节目标: 你能解释 `nn.Module -> forward -> loss -> backward -> optimizer.step` 这条线, 并知道网络参数是谁更新的。

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)
print(torch.__version__)

2.7.0+cu128


## Tensor shape

rsl_rl 里最常见的第一维是 batch。比如 `[num_envs, obs_dim]`。

In [ ]:
num_envs = 4
obs_dim = 10
obs = torch.randn(num_envs, obs_dim)

print('obs shape:', obs.shape)
print('requires_grad:', obs.requires_grad)

## 一个最小 critic

critic 的职责: 输入 observation, 输出 value。

In [ ]:
class TinyCritic(nn.Module):
    def __init__(self, obs_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 32),
            nn.ELU(),
            nn.Linear(32, 1),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs)

critic = TinyCritic(obs_dim)
values = critic(obs)
print('values shape:', values.shape)
print('first value:', values[0].item())

## loss.backward 和 optimizer.step

训练不是 magic。`backward()` 计算梯度, `step()` 改参数。

In [ ]:
target_returns = torch.randn(num_envs, 1)
optimizer = torch.optim.Adam(critic.parameters(), lr=1e-3)

before = critic.net[0].weight.detach().clone()

values = critic(obs)
loss = (values - target_returns).pow(2).mean()

optimizer.zero_grad()
loss.backward()
optimizer.step()

after = critic.net[0].weight.detach().clone()
print('loss:', loss.item())
print('first layer changed:', not torch.allclose(before, after))

## detach

PPO rollout 期间会 `detach()` action/value/log_prob。意思是保留数值, 切断梯度历史。

In [ ]:
x = torch.randn(4, 3, requires_grad=True)
y = x * 2.0
z = y.detach()

print('x requires_grad:', x.requires_grad)
print('y requires_grad:', y.requires_grad)
print('z requires_grad:', z.requires_grad)

## 作业

1. 把 hidden dim 从 32 改成 64, 预测参数数量会不会变。
2. 打印每个参数名和 shape: `for name, p in critic.named_parameters()`。
3. 用自己的话解释: 为什么 `detach()` 后不能 backward 到原来的 tensor。